# Output Parser

In [2]:
from langchain_classic.output_parsers import CommaSeparatedListOutputParser
from langchain_classic.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

In [3]:
model = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), temperature=0,model="gemini-2.5-flash")

<b>CSV Parser

In [4]:
output_paser = CommaSeparatedListOutputParser()

format_instruction = output_paser.get_format_instructions()
prompt = PromptTemplate(
    template="List five places {places}.\n{format_instruction}",
    input_variables=["places"],
    partial_variables={"format_instruction": format_instruction}
)

chain = prompt | model | output_paser

chain.invoke({"places": "for summer tourism in Pakistan"})

['Naran Kaghan', 'Skardu', 'Hunza Valley', 'Swat Valley', 'Murree']

<b>JSON Parser

This output parser allow users to specify a JSON schema and query LLmMs for outputs that conform to that schema. 

In [7]:
# using pydantic to declare data model
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

In [8]:
class Travel(BaseModel):
    place: str = Field(description="Name of the places")
    description: str = Field(description="Description of the place")
    activities: str = Field(description="What to do in that place")

In [15]:
# a query intended to prompt a LM to populate the data structure
travel_query = "Suggest a place in Pakistan for going to trip this summer avoid heat"
parser = JsonOutputParser(pydantic_object=Travel)

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instruction}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instruction": parser.get_format_instructions()},
)

chain = prompt | model | parser

chain.invoke({"query": travel_query})

{'place': 'Hunza Valley',
 'description': 'Nestled in the majestic Karakoram mountain range, Hunza Valley is a breathtakingly beautiful region in Gilgit-Baltistan, Pakistan. Known for its stunning landscapes, ancient forts, and vibrant culture, it offers a cool escape from the summer heat with its pleasant climate, lush greenery, and snow-capped peaks.',
 'activities': 'Explore the ancient Baltit and Altit Forts, hike to the scenic Passu Cones, visit Attabad Lake for boating, trek to the Batura Glacier, enjoy local cuisine, interact with the friendly locals, and capture panoramic views of Rakaposhi and other towering peaks.'}

<b>Without pydantic

In [ ]:
# a query intended to prompt a LM to populate the data structure
travel_query = "Suggest a place in Pakistan for going to trip this summer avoid heat"
parser = JsonOutputParser()

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instruction}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instruction": parser.get_format_instructions()},
)

chain = prompt | model | parser

chain.invoke({"query": travel_query})

<b>Structured Output Parser

this is used when we want to return multiple things.This is usefull for less powerful model

In [16]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

response_schema = [
    ResponseSchema(name="answer", description="Answer the user question"),
    ResponseSchema(name="description",description="Brief detail description on the answer topic"),
    ResponseSchema(
        name="applications",
        description="real world application of the answer topic"
    )
]
output_paser = StructuredOutputParser.from_response_schemas(response_schema)


In [19]:
format_instruction = output_paser.get_format_instructions()
prompt = PromptTemplate(
    template="answer the user question as best as possible.\n{format_instructions}.\n{question}",
    input_variables=["question"],
    partial_variables={"format_instructions": format_instruction},
)

In [20]:
chain = prompt | model | output_paser

chain.invoke({"question": "Name an invention in Healthcare that has caused revolution in twenty first century"})

{'answer': 'CRISPR-Cas9 Gene Editing Technology',
 'description': "CRISPR-Cas9 (Clustered Regularly Interspaced Short Palindromic Repeats and CRISPR-associated protein 9) is a revolutionary gene-editing tool discovered in the early 21st century (specifically, its application as a gene-editing tool was demonstrated in 2012). It allows scientists to precisely target and modify specific DNA sequences within a genome. This technology acts like molecular scissors, enabling the 'cutting' of DNA at desired locations, which can then be used to delete, insert, or replace genes with unprecedented accuracy and efficiency. It originated from a natural defense mechanism in bacteria against viruses.",
 'applications': "The applications of CRISPR-Cas9 are vast and continue to expand, fundamentally changing various fields:\n1.  **Curing Genetic Diseases:** Potential to correct genetic mutations responsible for diseases like sickle cell anemia, cystic fibrosis, Huntington's disease, and Duchenne muscul